# Station Stacking v3 - KAUS

Wide HRRR/GFS same-day 11am notebook for `KAUS`.

This version keeps the v2 notebook feature engineering and adds SDK-backed 11 AM high-so-far features from `observed_high_temp_through_as_of_f`. Artifacts are written to `data/calibration/station_stacking_v3`.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KAUS"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
from src.calibration.station_stacking import (
    StationStackingConfig,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## V3 Feature Engineering

Adds v2 features plus SDK-backed 11 AM high-so-far signals. The high-so-far feature is computed from same-day station observations at or before the 11 AM local as-of time.


In [3]:
import numpy as np
import pandas as pd
import src.calibration.station_stacking as station_stacking_module

V3_FEATURE_COLUMNS = [
    "v2_recent_heat_anomaly_f",
    "v2_recent_heat_momentum_f",
    "v2_morning_warmup_to_consensus_f",
    "v2_consensus_minus_7d_actual_f",
    "v2_spread_per_warmup_f",
    "v2_humidity_warmup_interaction",
    "v3_high_so_far_above_current_f",
    "v3_remaining_warmup_from_high_so_far_f",
    "v3_high_so_far_minus_lag_1d_f",
    "v3_high_so_far_minus_7d_actual_f",
    "v3_remaining_warmup_per_spread_f",
    "v3_humidity_remaining_warmup_interaction",
]


def add_v3_feature_engineering(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    observed_temp = pd.to_numeric(out.get("observed_temp_at_as_of_f"), errors="coerce")
    high_so_far = pd.to_numeric(out.get("observed_high_temp_through_as_of_f"), errors="coerce")
    observed_humidity = pd.to_numeric(out.get("observed_humidity_at_as_of"), errors="coerce")
    provider_mean = pd.to_numeric(out.get("provider_mean_high_f"), errors="coerce")
    provider_spread = pd.to_numeric(out.get("provider_spread_high_f"), errors="coerce")
    lag_1d = pd.to_numeric(out.get("actual_high_lag_1d"), errors="coerce")
    roll_7d = pd.to_numeric(out.get("actual_high_roll_7d_mean"), errors="coerce")
    roll_30d = pd.to_numeric(out.get("actual_high_roll_30d_mean"), errors="coerce")

    warmup_to_consensus = provider_mean - observed_temp
    remaining_warmup = provider_mean - high_so_far
    out["v2_recent_heat_anomaly_f"] = lag_1d - roll_30d
    out["v2_recent_heat_momentum_f"] = roll_7d - roll_30d
    out["v2_morning_warmup_to_consensus_f"] = warmup_to_consensus
    out["v2_consensus_minus_7d_actual_f"] = provider_mean - roll_7d
    out["v2_spread_per_warmup_f"] = provider_spread / warmup_to_consensus.abs().clip(lower=1.0)
    out["v2_humidity_warmup_interaction"] = (observed_humidity / 100.0) * warmup_to_consensus
    out["v3_high_so_far_above_current_f"] = high_so_far - observed_temp
    out["v3_remaining_warmup_from_high_so_far_f"] = remaining_warmup
    out["v3_high_so_far_minus_lag_1d_f"] = high_so_far - lag_1d
    out["v3_high_so_far_minus_7d_actual_f"] = high_so_far - roll_7d
    out["v3_remaining_warmup_per_spread_f"] = remaining_warmup / provider_spread.abs().clip(lower=1.0)
    out["v3_humidity_remaining_warmup_interaction"] = (observed_humidity / 100.0) * remaining_warmup
    return out


if not hasattr(station_stacking_module, "_v3_original_build_station_wide_dataset"):
    station_stacking_module._v3_original_build_station_wide_dataset = station_stacking_module.build_station_wide_dataset


def build_station_wide_dataset_v3(*args, **kwargs):
    frame = station_stacking_module._v3_original_build_station_wide_dataset(*args, **kwargs)
    return add_v3_feature_engineering(frame)


station_stacking_module.build_station_wide_dataset = build_station_wide_dataset_v3
V3_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction',
 'v3_high_so_far_above_current_f',
 'v3_remaining_warmup_from_high_so_far_f',
 'v3_high_so_far_minus_lag_1d_f',
 'v3_high_so_far_minus_7d_actual_f',
 'v3_remaining_warmup_per_spread_f',
 'v3_humidity_remaining_warmup_interaction']

## Model Scores


In [4]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_verbose=OPTUNA_VERBOSE,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3",
)
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-08 23:00:02,196] A new study created in memory with name: no-name-dd91c94c-ae8b-4e92-9ecd-d30caf3f9bf0
[I 2026-06-08 23:00:09,204] Trial 0 finished with value: 6.1537776520801994 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 6.1537776520801994.
[I 2026-06-08 23:01:24,830] Trial 1 finished with value: 5.933851356043147 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 5.933851356043147.
[I 2026-06-08 23:01:42,614] Trial

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,660,4.208539,6.019106
1,validation_2024_2025,lightgbm,660,3.865543,5.929835
2,validation_2024_2025,catboost,660,3.791782,5.734047
3,validation_2024_2025,hrrr_raw,660,4.139337,5.917629
4,validation_2024_2025,gfs_raw,660,4.616216,6.792032
5,test_2026,xgboost,137,4.274583,5.363051
6,test_2026,lightgbm,137,4.370841,5.506246
7,test_2026,catboost,137,3.968941,5.149760
8,test_2026,ridge_stack,137,4.096248,5.201524
9,test_2026,hrrr_raw,137,3.486340,4.885076


## Version Comparison


In [5]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,hrrr_raw,137,3.486340,4.885076,v2
1,test_2026,hrrr_raw,137,3.486340,4.885076,v3
2,test_2026,gfs_raw,137,3.656254,4.990693,v2
3,test_2026,gfs_raw,137,3.656254,4.990693,v3
4,test_2026,catboost,137,3.968941,5.149760,v3
5,test_2026,gfs_raw,102,3.971651,5.397913,v1
6,test_2026,catboost,137,4.006909,5.113676,v2
7,test_2026,hrrr_raw,102,4.018587,5.456511,v1
8,test_2026,ridge_stack,137,4.096248,5.201524,v3
9,test_2026,ridge_stack,137,4.202218,5.271690,v2


## 2026 Weather Brackets


In [6]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,137,4.274583,5.363051,8.759124
1,lightgbm,137,4.370841,5.506246,13.138686
2,catboost,137,3.968941,5.149760,14.59854
3,ridge_stack,137,4.096248,5.201524,13.868613
4,hrrr_raw,137,3.486340,4.885076,18.248175
5,gfs_raw,137,3.656254,4.990693,15.328467


In [7]:
import pandas as pd


def adjacent_brackets(bracket):
    if pd.isna(bracket):
        return []
    text = str(bracket).strip()
    if not text or "-" not in text:
        return []
    try:
        lower = int(text.split("-", 1)[0])
    except ValueError:
        return []
    return [
        f"{lower - 2}-{lower - 1}",
        f"{lower}-{lower + 1}",
        f"{lower + 2}-{lower + 3}",
    ]


bracket_3way = result.bracket_predictions.copy()

valid = bracket_3way["actual_bracket"].notna() & bracket_3way["predicted_bracket"].astype(str).str.strip().ne("")
bracket_3way = bracket_3way.loc[valid].copy()
bracket_3way["picked_brackets"] = bracket_3way["predicted_bracket"].map(adjacent_brackets)
bracket_3way["three_bracket_hit"] = bracket_3way.apply(
    lambda row: row["actual_bracket"] in row["picked_brackets"],
    axis=1,
)

three_bracket_accuracy = (
    bracket_3way
    .groupby("method", as_index=False)
    .agg(
        count=("three_bracket_hit", "size"),
        exact_bracket_accuracy_pct=("bracket_hit", lambda x: x.mean() * 100),
        three_bracket_accuracy_pct=("three_bracket_hit", lambda x: x.mean() * 100),
    )
    .sort_values("three_bracket_accuracy_pct", ascending=False)
)

three_bracket_accuracy


,method,count,exact_bracket_accuracy_pct,three_bracket_accuracy_pct
2,hrrr_raw,137,18.248175,54.014599
1,gfs_raw,137,15.328467,50.364964
5,xgboost,137,8.759124,40.875912
4,ridge_stack,137,13.868613,40.145985
3,lightgbm,137,13.138686,38.686131
0,catboost,137,14.59854,37.956204


In [8]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="test_2026"),
    ],
    ignore_index=True,
)

preds["predicted_high_rounded_up_f"] = np.ceil(pd.to_numeric(preds["predicted_high_f"], errors="coerce"))
preds["within_1f_after_round_up"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_up_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_up_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round_up", "size"),
        within_1f_count=("within_1f_after_round_up", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round_up", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period

,period,method,count,within_1f_count,within_1f_accuracy_pct
2,test_2026,hrrr_raw,137,45,32.846715
1,test_2026,gfs_raw,137,38,27.737226
0,test_2026,catboost,137,31,22.627737
3,test_2026,lightgbm,137,27,19.708029
4,test_2026,ridge_stack,137,27,19.708029
5,test_2026,xgboost,137,25,18.248175
9,validation_2024_2025,lightgbm,660,250,37.878788
6,validation_2024_2025,catboost,660,245,37.121212
10,validation_2024_2025,xgboost,660,210,31.818182
8,validation_2024_2025,hrrr_raw,660,209,31.666667
